# Objective 1 EDA Notebook

This notebook supports thesis Objective 1:
- Identify suitable dataset
- Investigate relationships between parameters and CI/CD build outcomes


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr

plt.style.use('ggplot')
DATA = '../data/processed/unified_build_dataset.csv'
OUT = '../outputs/objective1/notebook'
import os
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA)
df.head()

In [ ]:
print('Rows:', len(df))
print('Failure rate:', df['is_failed'].mean())
missing = df.isna().mean().sort_values(ascending=False)
missing.head(15)

In [ ]:
# Class distribution
cls = df['is_failed'].value_counts().sort_index()
plt.figure(figsize=(5,3))
plt.bar(['success(0)','failure(1)'], [cls.get(0,0), cls.get(1,0)])
plt.title('Build outcome distribution')
plt.tight_layout()
plt.savefig(f'{OUT}/class_distribution.png', dpi=200)
plt.show()

In [ ]:
features = ['duration_sec','commit_message_len','additions','deletions','total_changes','files_changed','run_attempt']
rows=[]
for f in features:
    x = pd.to_numeric(df[f], errors='coerce')
    y = pd.to_numeric(df['is_failed'], errors='coerce')
    m = x.notna() & y.notna()
    if m.sum() > 10 and y[m].nunique() > 1:
        c,p = pointbiserialr(x[m], y[m])
    else:
        c,p = np.nan, np.nan
    rows.append({'feature':f,'point_biserial_corr':c,'p_value':p})
assoc = pd.DataFrame(rows).sort_values(by='point_biserial_corr', key=lambda s: s.abs(), ascending=False)
assoc

In [ ]:
plt.figure(figsize=(8,4))
a = assoc.dropna()
plt.bar(a['feature'], a['point_biserial_corr'])
plt.axhline(0, color='black', lw=0.8)
plt.xticks(rotation=25, ha='right')
plt.title('Parameter-outcome association (Objective 1)')
plt.tight_layout()
plt.savefig(f'{OUT}/assoc_barplot.png', dpi=200)
plt.show()

In [ ]:
corr_input = df[features + ['is_failed']].apply(pd.to_numeric, errors='coerce')
corr = corr_input.corr(numeric_only=True)
plt.figure(figsize=(8,6))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation heatmap')
plt.tight_layout()
plt.savefig(f'{OUT}/corr_heatmap.png', dpi=200)
plt.show()

In [ ]:
# Save publication-ready tables
assoc.to_csv(f'{OUT}/assoc_table.csv', index=False)
missing.to_csv(f'{OUT}/missingness_profile.csv')
summary = []
summary.append('# Objective 1 Notebook Findings')
summary.append(f'- Total records: {len(df)}')
summary.append(f'- Failure rate: {df['is_failed'].mean():.4f}')
if len(assoc.dropna()) > 0:
    top = assoc.iloc[0]
    summary.append(f'- Strongest association: {top['feature']} (corr={top['point_biserial_corr']:.4f}, p={top['p_value']:.4g})')
Path = __import__('pathlib').Path
Path(f'{OUT}/notebook_findings.md').write_text('\n'.join(summary), encoding='utf-8')
print('Saved outputs under', OUT)